Here we train the Neural Network (MLP) with PyTorch for first round in cirriculum training. 

We uncompress the fitted extrapolation constants back to a time serie with the à-prior physics model that fitted it, then at a far time point, find the time serie that has a median penetration of the dataset.

In [45]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Iterable, Tuple
import matplotlib.pyplot as plt

In [ ]:
# Raw datasets from image processing. We do not consider them for now. 
roots_raw = {
    "Nozzle 1"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241003_HZ_Nozzle1",
    "Nozzle 2"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241017_HZ_Nozzle2",
    "Nozzle 3"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241014_HZ_Nozzle3",
    "Nozzle 4"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241007_HZ_Nozzle4",
    "Nozzle 5"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241010_HZ_Nozzle5",
    "Nozzle 6"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241011_HZ_Nozzle6",
    "Nozzle 7"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241015_HZ_Nozzle7",
    "Nozzle 8"  :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20241016_HZ_Nozzle8",
    "DS300"     :   r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\BC20220627 - Heinzman DS300 - Mie Top view"
}



In [ ]:
root_extraplotated = r"C:\Users\Jiang\Documents\Mie_Py\Mie_Postprocessing_Py\MLP\synthetic_data"

# Define a time step to compare penetration
defined_time_s = 5e-3 # 5ms


root_extraplotated = Path(root_extraplotated)

In [ ]:
# =============================================================================
# Utility Functions
# =============================================================================

def iter_files(directory: Path, suffix: str = None, keyword_match: str = None) -> Iterable[Path]:
    """Iterate files filtered by extension/suffix and optional filename keyword."""
    if not directory.exists():
        return []
    files = sorted(p for p in directory.iterdir() if p.is_file())
    if suffix:
        token = suffix.lower()
        if token.startswith("."):
            files = [f for f in files if f.suffix.lower() == token]
        else:
            files = [f for f in files if f.name.lower().endswith(token)]
    if keyword_match:
        key = keyword_match.lower()
        files = [f for f in files if key in f.name.lower()]
    return files

In [31]:
# Stuff copied from fit_raw_data
from scipy.special import expit

MIN_TI = 0.0

def spray_penetration_model_sigmoid(params, t):
    """
    params (log-space): [log_k_sqrt, log_k_quarter, log_t0, log_s]
    """
    log_k_sqrt, log_k_quarter, log_t0, log_s = params

    k_sqrt = np.exp(log_k_sqrt)
    k_quarter = np.exp(log_k_quarter)
    t0 = np.exp(log_t0) + MIN_TI
    s = np.exp(log_s)

    t = np.clip(np.asarray(t, dtype=float), 1e-9, None)
    sqrt_segment = k_sqrt * np.sqrt(t)
    quarter_root_segment = k_quarter * np.power(t, 0.25)
    w = expit((t - t0) / s)

    return (1.0 - w) * sqrt_segment + w * quarter_root_segment


In [ ]:
# Get all folders containing the cleaned data.
folders  = []

for p in root_extraplotated.iterdir():
    # Must be folder not file & etc
    if not p.is_dir():
        continue
    
    # Find the /clean/ subdirs
    direct_clean = p / "clean"
    if direct_clean.is_dir():
        folders.append(direct_clean)
        continue



'BC20220627 - Heinzman DS300 - Mie Top view'

In [ ]:
rows = []  # 用来收集每个文件挑出的那一行 Collect the selected row from each file

for folder in folders:
    # Handle DS300 files later on.
    if folder.parts[-2] != "BC20220627 - Heinzman DS300 - Mie Top view":
        # print(folder)
        files = iter_files(folder, suffix=".csv")

        
        for file in files: 
            # Read the file
            df = pd.read_csv(file)
            
            # Extrat fitted constants
            log_k_sqrt      = df["log_k_sqrt"].to_numpy()
            log_k_quarter   = df["log_k_quarter"].to_numpy()
            log_s           = df["log_s"].to_numpy()
            log_t0          = df["log_t0"].to_numpy()
            
            # initiate an array to collect penetrations
            penetrations = np.zeros(len(df))
            
            # Compute penetration at given time
            for i in range(len(df)):
                penetrations[i] = spray_penetration_model_sigmoid([log_k_sqrt[i], log_k_quarter[i], log_t0[i], log_s[i]], defined_time_s)

            # Find the row entry where the penetration at this time is the closest to median
            median_pen = np.nanmedian(penetrations)
            i_closest = np.nanargmin(np.abs(penetrations - median_pen))

            # Get the row and turn into dict for the ease of accumulation
            row = df.iloc[i_closest].to_dict()

            # Add tracing infos
            row["source_file"] = str(file)
            row["source_folder"] = str(folder)
            row["defined_time_s"] = defined_time_s
            row["penetration_at_defined_time"] = penetrations[i_closest]
            row["median_penetration_in_file"] = median_pen

            rows.append(row)

# Generate df
df_collection = pd.DataFrame(rows).reset_index(drop=True)



median = 148.92287288236935
closest position = 36
df index label = 36
